# Feature Engineering — K. pneumoniae + Meropenem MIC Creep
## Vivli AMR Surveillance Challenge 2026

**Input**: ATLAS raw data (parsed from `04_atlas_eda.ipynb`)
**Output**: `data/processed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

**Sections**
1. Load & parse ATLAS data
2. Specimen type mapping
3. Build feature matrix
4. Censoring rate by split (validation)
5. Target distribution — train vs test
6. Feature correlation heatmap
7. Save processed data
8. Feature summary

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

project_root = Path('../')
sys.path.insert(0, str(project_root / 'src'))

from data.loader import ATLASLoader
from data.preprocessor import MICPreprocessor
from features.engineer import (
    CARBAPENEMASE_GENES, TRAIN_END, TEST_START, TEST_END,
    build_features, build_target, map_specimen, run_pipeline, time_split,
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (13, 5), 'figure.dpi': 100})

DATA_DIR  = project_root / 'data' / 'raw'
OUT_DIR   = project_root / 'data' / 'processed'
REPORTS   = project_root / 'reports'
EUCAST_R  = 8

WOUND_SOURCES   = {'Wound', 'Abscess', 'Skin and Soft Tissue'}
AGE_GROUP_BROAD = {
    '0 - 17':  'Paediatric (0-17)',
    '18 - 30': 'Adult (18-60)',
    '31 - 60': 'Adult (18-60)',
    '61+':     'Elderly (61+)',
}

## 1. Load & parse ATLAS data

In [ ]:
loader = ATLASLoader(DATA_DIR)
df_raw = loader.load('Klebsiella pneumoniae', antibiotic='Meropenem')

df = df_raw.copy()
parsed = df['Meropenem'].apply(
    lambda x: MICPreprocessor.parse_censored_mic(x) if pd.notna(x) else (None, None)
)
df['mic_value']    = parsed.apply(lambda t: t[0])
df['mic_operator'] = parsed.apply(lambda t: t[1])
df = df[df['mic_value'].notna() & (df['mic_value'] > 0)].copy()

df['mic_log2']     = np.log2(df['mic_value'])
df['is_censored']  = df['mic_operator'].isin(['>', '<', '>=', '<='])
df['is_resistant'] = df['mic_value'] >= EUCAST_R
df['age_group_broad'] = df['Age Group'].map(AGE_GROUP_BROAD).fillna('Adult (18-60)')
df['military_proxy'] = (
    df['Source'].isin(WOUND_SOURCES)
    & (df['Gender'] == 'Male')
    & (df['Age Group'].isin(['18 - 30', '31 - 60']))
)

for gene in CARBAPENEMASE_GENES:
    if gene in df.columns:
        s = df[gene].astype(str).str.strip()
        df[f'{gene}_pos'] = df[gene].notna() & ~s.isin(['', 'nan', '0', 'None'])

print(f"Rows: {len(df):,}  |  Years: {df['Year'].min()}–{df['Year'].max()}")
print(f"Censored: {df['is_censored'].mean()*100:.1f}%  |  Resistant: {df['is_resistant'].mean()*100:.1f}%")

## 2. Specimen type mapping

ATLAS `Source` has 100+ free-text values. We map them to 5 broad clinical categories
using substring matching. `other` is the reference category in the model.

In [ ]:
df['specimen_broad'] = df['Source'].apply(map_specimen)

print('Specimen mapping coverage:')
counts = df['specimen_broad'].value_counts()
for cat, n in counts.items():
    print(f'  {cat:<15} {n:>7,}  ({n/len(df)*100:.1f}%)')

# Spot check — what raw sources map to each category?
print('\nTop raw values per category:')
for cat in counts.index:
    top = df[df['specimen_broad'] == cat]['Source'].value_counts().head(5).index.tolist()
    print(f'  {cat}: {top}')

## 3. Build feature matrix

**Time split**: train ≤ 2018, test 2019–2022. **Never shuffle** — future data must not leak into training.

| Feature group | Columns |
|---|---|
| Temporal | `year` |
| Demographics | `gender_male`, `age_paediatric`, `age_elderly` |
| Clinical | `military_proxy`, `spec_*` (OHE, other=reference) |
| Geographic | `ctry_*` (OHE, drop_first) |
| Genes | `KPC_pos` … `GES_pos` |
| Quality | `is_censored`, `pct_censored_year` |

**Target**: `log2(mic_value)`

In [ ]:
train_df, test_df = time_split(df)

X_train = build_features(train_df)
y_train = build_target(train_df)
X_test  = build_features(test_df)
y_test  = build_target(test_df)

print(f'Train: {len(X_train):,} rows  ({df["Year"].min()}–{TRAIN_END})')
print(f'Test:  {len(X_test):,} rows  ({TEST_START}–{TEST_END})')
print(f'Features: {X_train.shape[1]}')
print(f'\nColumn groups:')
for prefix, label in [('year|gender|age|military|is_censored|pct_', 'Core'),
                       ('spec_', 'Specimen'), ('ctry_', 'Country'), ('_pos', 'Genes')]:
    cols = [c for c in X_train.columns if any(c.startswith(p) or c.endswith(p.lstrip('_pos')) for p in [prefix])]
core = [c for c in X_train.columns if not c.startswith('spec_') and not c.startswith('ctry_') and not c.endswith('_pos')]
spec = [c for c in X_train.columns if c.startswith('spec_')]
ctry = [c for c in X_train.columns if c.startswith('ctry_')]
gene = [c for c in X_train.columns if c.endswith('_pos')]

print(f'  Core ({len(core)}):     {core}')
print(f'  Specimen ({len(spec)}): {spec}')
print(f'  Genes ({len(gene)}):    {gene}')
print(f'  Country ({len(ctry)}):  {len(ctry)} columns')

## 4. Censoring rate by split

Validates that the `pct_censored_year` feature captures the methodology artifact
(the 85%→30%→85% pattern documented in the EDA).

In [ ]:
quality = df.groupby('Year')['is_censored'].mean().reset_index()
quality['pct_censored'] = quality['is_censored'] * 100

fig, ax = plt.subplots()
train_q = quality[quality['Year'] <= TRAIN_END]
test_q  = quality[(quality['Year'] >= TEST_START) & (quality['Year'] <= TEST_END)]

ax.bar(train_q['Year'], train_q['pct_censored'], color='#1f77b4', alpha=0.7, label='Train')
ax.bar(test_q['Year'],  test_q['pct_censored'],  color='#d62728', alpha=0.7, label='Test')
ax.set_ylabel('% Censored MIC values', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title('Censoring Rate by Year — Train vs Test Split', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS / 'censoring_by_split.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Train mean censoring: {train_q['pct_censored'].mean():.1f}%")
print(f"Test mean censoring:  {test_q['pct_censored'].mean():.1f}%")

## 5. Target distribution — train vs test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, subset, label, color in [
    (axes[0], train_df, f'Train ({df["Year"].min()}–{TRAIN_END})', '#1f77b4'),
    (axes[1], test_df,  f'Test ({TEST_START}–{TEST_END})',         '#d62728'),
]:
    ax.hist(subset['mic_log2'], bins=40, color=color, alpha=0.8, edgecolor='white', lw=0.3)
    ax.axvline(np.log2(EUCAST_R), color='black', linestyle='--', lw=1.5,
               label=f'R breakpoint (log2={np.log2(EUCAST_R):.0f})')
    ax.set_xlabel('log2(MIC)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

    med  = subset['mic_log2'].median()
    mean = subset['mic_log2'].mean()
    pct_r = (subset['mic_log2'] >= np.log2(EUCAST_R)).mean() * 100
    print(f'{label}: median={med:.2f}  mean={mean:.2f}  %R={pct_r:.1f}%')

plt.suptitle('Target Distribution (log2 MIC) — Train vs Test', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature correlation heatmap (core + gene features)

Country and specimen dummies excluded for readability. Key things to look for:
- High correlation between `year` and `pct_censored_year` → confirms censoring is time-confounded
- Gene flags vs `log2_mic` → which genes are most predictive of high MIC

In [ ]:
core_cols = ['year', 'gender_male', 'age_paediatric', 'age_elderly',
             'military_proxy', 'is_censored', 'pct_censored_year']
gene_cols = [c for c in X_train.columns if c.endswith('_pos')]

corr_df = X_train[core_cols + gene_cols].copy()
corr_df['log2_mic'] = y_train.values
corr = corr_df.corr()

size = len(corr_df.columns)
fig, ax = plt.subplots(figsize=(size * 0.75 + 1, size * 0.75 + 1))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title('Feature Correlation (core + gene flags) — Train set',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with target — sorted
print('\nCorrelation with log2(MIC) — sorted:')
target_corr = corr['log2_mic'].drop('log2_mic').sort_values(key=abs, ascending=False)
for feat, r in target_corr.items():
    print(f'  {feat:<25} {r:+.3f}')

## 7. Save processed data

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_parquet(OUT_DIR / 'X_train.parquet')
y_train.to_frame().to_parquet(OUT_DIR / 'y_train.parquet')
X_test.to_parquet(OUT_DIR / 'X_test.parquet')
y_test.to_frame().to_parquet(OUT_DIR / 'y_test.parquet')

print('Saved:')
for fname in ['X_train.parquet', 'y_train.parquet', 'X_test.parquet', 'y_test.parquet']:
    size_kb = (OUT_DIR / fname).stat().st_size / 1024
    print(f'  {fname:<25} {size_kb:.0f} KB')

## 8. Feature summary table

In [ ]:
summary = pd.DataFrame({
    'mean':    X_train.mean(),
    'std':     X_train.std(),
    'min':     X_train.min(),
    'max':     X_train.max(),
    'n_unique': X_train.nunique(),
})

print('Core features:')
core_cols = ['year', 'gender_male', 'age_paediatric', 'age_elderly',
             'military_proxy', 'is_censored', 'pct_censored_year']
print(summary.loc[[c for c in core_cols if c in summary.index]].to_string(float_format='{:.3f}'.format))

print('\nGene flags (train set):')
gene_cols = [c for c in X_train.columns if c.endswith('_pos')]
print(summary.loc[gene_cols, ['mean', 'n_unique']].rename(columns={'mean': 'prevalence'}).to_string(float_format='{:.3f}'.format))

print(f'\nTotal features: {X_train.shape[1]}')
print(f'  Core:     {len(core_cols)}')
print(f'  Specimen: {len([c for c in X_train.columns if c.startswith("spec_")])}')
print(f'  Country:  {len([c for c in X_train.columns if c.startswith("ctry_")])}')
print(f'  Genes:    {len(gene_cols)}')
print('\nNext step → 06_model_training.ipynb')